In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers.models.gpt2.modeling_gpt2 import GPT2Attention
import torch.nn as nn

In [ ]:
def get_target_linear_layers(model):
    target_layers = []
    for name, module in model.named_modules():
        if ".attn.c_attn" in name or ".attn.c_proj" in name:
            target_layers.append((name, module))
    return target_layers

In [ ]:
def compute_svd_rank(weight: torch.Tensor, energy_threshold: float = 0.9) -> int:
    if weight.ndim > 2:
        weight = weight.view(weight.shape[0], -1)
    
    _, S, _ = torch.linalg.svd(weight.cpu(), full_matrices=False)   
    total_energy = torch.sum(S**2)
    cumulative_energy = torch.cumsum(S**2, dim=0)
    
    r = torch.searchsorted(cumulative_energy, energy_threshold * total_energy).item() + 1
    return r

In [ ]:
model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.resize_token_embeddings(len(tokenizer))

In [ ]:
layer_ranks = {}

for name, layer in get_target_linear_layers(model):
    weight = layer.weight.data
    r = compute_svd_rank(weight, energy_threshold=0.9)
    layer_ranks[name] = r
    print(f"{name}: r = {r} (of {weight.shape})")

In [ ]:
sum(layer_ranks.values())/(768*len(layer_ranks))*100

In [ ]:
class Sketch:

  def __init__(self, A, k, l):
    self.A = A
    m = A.size(0)
    n = A.size(1)
    Omega = torch.randn(n, k)
    Psi = torch.randn(l, m)
    self.Omega, _ = torch.linalg.qr(Omega)
    Psi_T, _ = torch.linalg.qr(Psi.T)
    self.Psi = Psi_T.T
    self.Y = A @ self.Omega
    self.W = self.Psi @ A

  def linear_update(self, H, theta, eta):
    self.Y = theta*self.Y + eta*H @ self.Omega
    self.W = theta*self.W + eta*self.Psi @ H

  def low_rank_approx(self):
    self.Q, _ = torch.linalg.qr(self.Y, mode='reduced')
    U, T = torch.linalg.qr(self.Psi @ self.Q, mode='reduced')
    self.X = torch.linalg.solve(T, U.T @ self.W)
    return self.Q, self.X

  def get_original(self):
    return self.A

  def get_sketch(self):
    return self.Y, self.W

  def get_test(self):
    return self.Omega, self.Psi

  def get_approx(self):
    return self.Q, self.X

In [ ]:
def replace_linear_with_sketch(layer: nn.Linear, rank: int) -> nn.Sequential:
    W = layer.weight.data
    
    s = Sketch(W, rank, rank)
    Q, X = s.low_rank_approx()

    second = nn.Linear(Q.shape[1], rank, bias=False)
    first = nn.Linear(rank, X.shape[0], bias=False)

    second.weight.data = Q.T
    first.weight.data = X.T

    return nn.Sequential(first, second)

In [ ]:
decomposed_layers = []
for name, module in model.named_modules():
    if ".attn.c_attn" in name or ".attn.c_proj" in name:
        rank = layer_ranks.get(name, None)
        if rank is not None:
            print(f"Reemplazando {name} con rank {rank}")
            decomposed_layers.append(replace_linear_with_sketch(module, rank))

In [ ]:
actual = 0
for name, module in model.named_modules():
    if isinstance(module, GPT2Attention):
        module.c_attn = decomposed_layers[actual]
        module.c_proj = decomposed_layers[actual + 1]
        actual += 2
        

In [ ]:
# Prueba del modelo
article_text = """El presidente dio una conferencia sobre política internacional, abordando temas clave como comercio exterior, relaciones diplomáticas con Asia y la crisis humanitaria en Europa del Este."""

# Formar el prompt como lo entrenaste:
prompt = f"Article: {article_text.strip()}\n\nSummary:"

# Tokenizar
inputs = tokenizer(prompt, return_tensors="pt")

# Generar texto (usa CPU o GPU si está disponible)
output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95,
    num_return_sequences=1
)

# Decodificar
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

In [ ]:
from datasets import load_dataset

# Cargar el dataset
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train")

# Concatenar artículo y resumen como texto
def format_for_causal_lm(example):
    return {
        "text": f"Article: {example['article']}\n\nSummary: {example['highlights']}"
    }

formatted_dataset = dataset.map(format_for_causal_lm)

In [ ]:
def tokenize_function(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_datasets = formatted_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

In [ ]:
# 1% del dataset
k = round(len(tokenized_datasets)*0.01)
print(k)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-sketch",        # carpeta temporal de salida
    per_device_train_batch_size=2,
    num_train_epochs=1,
    save_strategy="epoch",          # solo guarda al final de cada época
    save_total_limit=1,             # solo guarda el último
    logging_steps=100,
    report_to="none",
    #fp16=True,                      # opcional: solo si tu GPU lo soporta
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
    tokenizer=tokenizer,
)

In [ ]:
trainer.train()

In [ ]:
from huggingface_hub import login, HfApi
import os
#from google.colab import userdata

HF_TOKEN = "" #userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
repo_id = "Pseudokiwi/gpt2-sketch"

In [ ]:
api = HfApi()
api.create_repo(repo_id=repo_id, private=True)

In [ ]:
api.update_repo_settings(repo_id="Pseudokiwi/gpt2-sketch", private=True)

In [ ]:
trainer.push_to_hub()